# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = **one pseudonymized content item** (identified by `content_id`) for a specific client (`client_id`). The data is aggregated over a **trailing 90-day time window**, with some comparison metrics aggregated over the last 30 days and the previous 30 days.

In [1]:
import pandas as pd
import duckdb

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
con = duckdb.connect()
con.register('df', df)

# Verify grain: count of rows where content_id + client_id is not unique should be 0
grain_check = con.execute("""
    SELECT content_id, client_id, COUNT(*) as c
    FROM df
    GROUP BY content_id, client_id
    HAVING c > 1
""").df()

print(f"Grain violations: {len(grain_check)} rows")
print(f"Total rows in dataset: {len(df)}")

Grain violations: 0 rows
Total rows in dataset: 30000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Context:** `content_id`, `client_id` (Used for grouping and joining; never for modeling).
- **Feature:** `search_volume`, `competition`, `cpc`, `content_type`, `main_intent`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update` (Knowable signals used to analyze ranking and CTR).
- **Label / proxy:** `avg_position`, `ctr`, `engagement_rate`, `scroll_rate` (These are the observed outcomes we are running our signal analysis against).
- **Excluded:** 
  - `trend_direction` & `trend_pct`: Derived labels based on trend calculations; we are not predicting decline/growth directly here.
  - `is_declining_label`: Artificial label derived from `trend_direction`; not an observed outcome.
  - `provider_used` & `model_used`: Future/Product-decision flags that may be incomplete or irrelevant for standard SEO signal analysis unless specifically isolating AI content.

In [2]:
# Verify basic schema and field presence
print("Context columns check:", all(c in df.columns for c in ['content_id', 'client_id']))
print("Feature columns check:", all(c in df.columns for c in ['search_volume', 'competition', 'word_count', 'content_age_days']))
print("Label columns check:", all(c in df.columns for c in ['avg_position', 'ctr', 'engagement_rate']))

Context columns check: True
Feature columns check: True
Label columns check: True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


In [3]:
# Missing values check by column
missing_check = con.execute("""
    SELECT 
        AVG(CASE WHEN word_count IS NULL THEN 100.0 ELSE 0 END) AS pct_missing_word_count,
        AVG(CASE WHEN search_volume IS NULL THEN 100.0 ELSE 0 END) AS pct_missing_search_volume,
        AVG(CASE WHEN avg_position = 0 THEN 100.0 ELSE 0 END) AS pct_zero_position
    FROM df
""").df()

print("Missing Value / Zero Checks (%):")
print(missing_check)

# Missingness by content type (verifying 'missingness follows content_type')
missing_by_type = con.execute("""
    SELECT 
        content_type,
        COUNT(*) as total_rows,
        AVG(CASE WHEN word_count IS NULL THEN 100.0 ELSE 0 END) AS pct_missing_word_count
    FROM df
    GROUP BY content_type
    ORDER BY pct_missing_word_count DESC
""").df()

print("\nMissing word_count by Content Type:")
print(missing_by_type)

Missing Value / Zero Checks (%):
   pct_missing_word_count  pct_missing_search_volume  pct_zero_position
0               25.663333                   8.226667           4.016667

Missing word_count by Content Type:
         content_type  total_rows  pct_missing_word_count
0     keyword article       27207               28.297865
1  comparison article         697                0.000000
2      feedly article        2096                0.000000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. **`avg_position = 0` means 'no data', not rank 0:** We have around 4% of rows where the position is zero. These rows cannot be used safely in correlation studies for ranking impact.
2. **Missingness follows `content_type`:** For some content types (like short-form or specific categories), `word_count` is highly missing. Blindly filling this with `0` will inject a category signal into a numerical feature. We should create `has_word_count` flags.
3. **Different Measurement Systems:** Rates like `scroll_rate` and `ai_traffic_pct` can exceed 100% because the numerator and denominator come from different analytics systems. This means they are not true "percentages" bound by 0-100, but rather index scores.
4. **No Historical Snapshotting:** This dataset is a snapshot of trailing windows, not a daily time-series. We cannot evaluate exactly *when* an item started declining, only its aggregated state over the 90-day window.

In [4]:
# Verify 'exceeding 100%' claim
exceeding_check = con.execute("""
    SELECT 
        SUM(CASE WHEN scroll_rate > 100 THEN 1 ELSE 0 END) AS scroll_rate_gt_100,
        SUM(CASE WHEN ai_traffic_pct > 100 THEN 1 ELSE 0 END) AS ai_traffic_pct_gt_100
    FROM df
""").df()

print("Rows with rates exceeding 100%:")
print(exceeding_check)

Rows with rates exceeding 100%:
   scroll_rate_gt_100  ai_traffic_pct_gt_100
0               119.0                   23.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.